# Computational Astrophysics — Homework Set 3 (10 pt)
## Numerical Errors, Derivatives, and Integration

**Topics:** Floating-point error, truncation error, Finite-difference derivatives, and Numerical quadrature  

**Instructions:** Read the descriptor for each problem, then write your solution code in the Jupyter code cell. Add your homework 3 assignment to the list of things to modify with `git add HW3.ipynb`, and commit changes in your assignment as frequently as you save your homework assignment. Once you are done, commit your finishing changes with the command `git commit -m "Completed HW 3"`, then push/submit to the instructor/TA using `git push`. If you push your assignment more than once, this is fine! Your most recently pushed assignment will be the one that is graded, just explain in the push that the most recent one you would like to be graded with a commit like `"Completed HW3"`, `"submitted HW3"`, `"submit"`, etc.

---

## Problem 1 — Floating point and truncation error in stellar atmospheres (3 pt)

The pressure at a given location in a star's interior is extremely important for determining the star's properties: its temperature, density, composition, nuclear reaction rate, etc. Hydrostatic equilibrium dictates how pressure varies with depth. In the interior of the sun, the pressure scale height $H_P$ is the distance over which pressure drops by a factor of $e$:

$$H_P = -\frac{dr}{d\ln P} = -P \left(\frac{dP}{dr} \right)^{-1}$$

For a star in hydrostatic equilibrium, $dP/dr = -\rho g$, so:

$$H_P = \frac{P}{\rho g}$$

At the base of the sun's photosphere, $P_0 \approx 1.2 \times 10^5$ dyn cm$^{-2}$, $\rho_0 \approx 2.0 \times 10^{-7}$ g cm$^{-3}$, $g_\odot = 2.74 \times 10^4$ cm s$^{-2}$, giving $H_P \approx 2.2 \times 10^7$ cm $\approx 220$ km.

To a pretty good approximation, $H_P$ is constant in the upper regions of the sun's atmosphere. Solving $dP/dr = -P/H_P$ then gives

$$P(r) = P_0 \exp\!\left(-\frac{r}{H_P}\right)$$

and re-arranging our formulae, we find an exact solution of

$$\frac{dP}{dr} = -\frac{P_0}{H_P} \exp\left(-\frac{r}{H_P}\right).$$ 

We take $r=0$ to be the base of the sun's photosphere

### Problem 1(a) - Exact and numerical derivative functions (1 pt)

Write two functions, one called `dPdr_exact`, which calculates the derative from the exact derivative of $P(r)$, or

$$ \left( \frac{dP}{dr} \right)_{\rm exact} = -\frac{P_0}{H_P} \exp\left(-\frac{r}{H_P}\right),$$ 

and another function `dPdr_num`, which calculates the numerical derivative from centered differencing, or

$$ \left( \frac{dP}{dr} \right)_{\rm num} = \frac{P(r+h) - P(r-h)}{2h}$$.

Have `dPdr_exact` take $r$, and `dPdr_num` take $h$ and $r$, as input parameters, with each output being the analytic or numerical derivative. Have the outputs be in cgs units of dyn cm$^{-3}$.

In [ ]:
# Code for 1a here


### Problem 1(b) - Plotting truncation versus roundoff error (1 pt)

As we discussed in class, making $h$ smaller and smaller does not necissarily make $dP/dr|_{\rm num}$ more and more accurate. When $h$ is large, $dP/dr|_{\rm num}$ is dominated by truncation error, and when $dP/dr|_{\rm num}$ is small, $dP/dr|_{\rm num}$ is dominated by roundoff error.

Define a function `error_deriv` that calculates the error in the numerical derivative,
$$ \left| \left( \frac{dP}{dr} \right)_{\rm num} - \left( \frac{dP}{dr} \right)_{\rm exact} \right| $$
with function inputs being $h$ and $r$. Plot this error as a function of $h/H_P$, with $h$ varied logarithmically between $10^{-9} H_P$ to $10^{-1} H_P$, for $r$ with the three different values of $r = 0 H_P, 1 H_P, 3 H_P$. Label each curve with its associated $r$ value using an f-string, and include the legend in the plot.

In [ ]:
# Code for 1b here


### Problem 1(c) - Discuss trunctation versus roundoff error (1 pt)

Our plot should show that at large versus small $h$ yields a numerical derivative dominated by truncation versus roundoff error. Discuss the range of $h$ values where one type of error dominates over another. How does this result depend on where the derivative is evaluated?

**Explanation for 1c here**


## Problem 2 - Numerical integrals and derivatives, galaxy rotation curves, and dark matter (3 pt)

One of the most compelling pieces of evidence for dark matter comes from galaxy rotation curves. In a galaxy where all mass is located in the visible stars and gas, we expect the gravity from this mass to cause gas in the galaxy to travel with a circular velocity

$$v_c(R) = \sqrt{\frac{GM(R)}{R}}$$

where $M(R)$ is the total mass enclosed within radius $R$. For the luminous disk, most mass is concentrated in the inner few kiloparsecs, and you expect the velocity to fall off. However, the velocity remains roughly constant. This is argued to imply the existance of unseen matter in the galaxy, called dark matter.

Below, we import the velocity rotation curve of NGC 3198 from Begeman (1989). The rotation curve is given in units of radial distance in arcminutes in the first column, and km/s for the rotational velocity of NGC 3198 in the second column. We convert arcminutes to the radial distance in kpc, using the distance of NGC 3198 (9.4 Mpc), given as `R_kpc` in the code below. We also give some other constants in cgs units that might be useful in your calculations.

In [1]:
import numpy as np

# Table 2 from Begeman (1989): rotation curve of NGC 3198
# Column 0: R (arcmin), Column 1: V_c (km/s)
rotcurve = np.array([
    [0.25,  55],
    [0.50,  92],
    [0.75, 110],
    [1.00, 123],
    [1.25, 134],
    [1.50, 142],
    [1.75, 145],
    [2.00, 147],
    [2.25, 148],
    [2.50, 152],
    [2.75, 155],
    [3.00, 156],
    [3.50, 157],
    [4.00, 153],
    [4.50, 153],
    [5.00, 154],
    [5.50, 153],
    [6.00, 150],
    [6.50, 149],
    [7.00, 148],
    [7.50, 146],
    [8.00, 147],
    [8.50, 148],
    [9.00, 148],
    [9.50, 149],
    [10.00, 150],
    [10.50, 150],
    [11.00, 149],
])

R_arcmin = rotcurve[:, 0]
V_c_kms = rotcurve[:, 1]
R_kpc = R_arcmin * 2.73  # at D = 9.4 Mpc


# Useful constants in cgs
G = 6.674e-8               # Newton's gravitational constant in cgs
kpc = 3.086e21             # kpc in cm
M_sun = 1.988e33           # Solar mass in g
km = 1e5                   # km in cm

### Problem 2(a) - Enclosed visible and total mass (1 pt)

Using relations between luminosity and total mass of galaxies, as well as the mass in HII gas, one can estimate that the total mass in visible matter within NGC 3198 with distance, which roughly goes as

$$ M_{\rm vis}(R) \approx M_\star \left[1 - e^{-R/R_d}\left(1 + \frac{R}{R_d}\right)\right].$$

Begelman (1989) set $R_d = 2.63$ kpc in NGC 3198, and find total mass in stars (and gas) to be roughly $M_\star \approx 3 \times 10^{10} \ M_\odot$. 

Plot the radial dependence of the mass in visible matter $M_{\rm vis}(R)$, versus the total mass in the galaxy inferred through the galactic rotation curve $M(R)$. Plot the distance on the x-axis in kpc on a linear scale, and the total mass in $M_\odot$ on the y-axis on a logarithmic scale. How do the two compare, and where are the two most discrepant?

In [ ]:
# Code for 2a here


**Explanation for 2a here**


### Problem 2(b) - Density using numerical derivatives (1 pt)

Assuming for simplicity that matter enclosed within a radius $r$ is spherically symmetric, its total mass up to $r$ is
$$ M(R) = \int^R 4\pi \rho(R') {R'}^2 d R' $$
Taking the derivative of both sides, one can find the density with radius
$$ \rho(R) = \frac{1}{4\pi R^2} \frac{d M}{d R}. $$

Write a function that calculates the density of matter $\rho$, given an input radial profile $R$ and enclosed mass profile $M(R)$. Use the central difference scheme to calculate the first derivative of the mass. For simplicity, if your radial grid goes from $i = 0, 1, 2, \dots, N-3, N-2, N-1$, calculate the density only at $i = 1, 2, \dots, N-2$, and ignore the density at the end-points.

In [ ]:
# Code for 2b here


### Problem 2(c) - Density of visible versus dark matter (1 pt)

Use the total enclosed mass, and estimated visible mass, to calculate the density of visible matter

$$ \rho_{\rm vis} = \frac{1}{4\pi R^2} \frac{d M_{\rm vis}}{dR}, $$

and the density of dark matter,

$$ \rho_{\rm DM} = \frac{1}{4\pi R^2} \frac{d M_{\rm DM}}{dR}, $$

where the enclosed mass in dark matter
$$ M_{\rm DM} = M - M_{\rm vis}. $$

Plot the density of  y-axis logarithmicaly in units of [$M_\odot$/kpc${}^3$], so that the result is plotted in more physically-understandable units. Plot radial distance on the x-axis in units of kpc. Label $\rho_{\rm vis}$ versus $\rho_{\rm DM}$, with the result displayed in the legend. Label the x and y-axes as well.

In [ ]:
# Code for 2c here


## Problem 3 — Gravitational Wave Inspiral Time: The Peters (1964) Integral (4 pt)

For LIGO to detect gravitational waves from a compact object binary, they have to shrink to very short seperations. A binary system of two compact objects (neutron stars, black holes, or white dwarfs) loses energy and angular momentum to gravitational wave radiation, causing the orbit to shrink and the eccentricity to evolve. One way binaries can merge faster is through a so-called "dynamical channel," where the eccentricity of the orbit is excited through some process, causing the binary to merge over a much shorter time-scale. In this problem, we will calculate how large eccentricities shortens this merger timescale.

Peters (1964) derived the time for such a binary to merge from an initial semi-major axis $a_0$ and eccentricity $e_0$:

$$T_{\rm merge} = \frac{12}{19}\frac{c_0^4}{\beta}\int_0^{e_0} \frac{e^{29/19}\left(1 + \frac{121}{304}e^2\right)^{1181/2299}}{(1-e^2)^{3/2}}\,de$$

The parameters $c_0$ and $\beta$ are defined as:

$$\beta = \frac{64}{5}\frac{G^3 m_1 m_2(m_1+m_2)}{c^5}$$

$$c_0 = \frac{a_0(1-e_0^2)}{e_0^{12/19}} \left(1 + \frac{121}{304}e_0^2\right)^{-870/2299}$$

where $m_1$, $m_2$ are the component masses, $G$ is Newton's constant, and $c$ is the speed of light.

In this problem, we will evaluate this integral to calculate the merger time for a neutron star binary, with masses $m_1 = m_2 = 1.4 M_\odot$, and initial semi-major axis $a_0 = 0.1 \ {\rm AU}$. Below, we define constants that you might need in cgs units.

In [ ]:
# CGS physical constants
G_cgs   = 6.674e-8      # Newton's gravitational constant in cgs [dyn cm^2 g^-2]
c_cgs   = 2.998e10      # speed of light in cm/s
M_sun_g = 1.989e33      # solar mass in g
AU_cm   = 1.496e13      # 1 AU in cm
yr_s    = 3.156e7       # seconds per year
Gyr_s   = yr_s * 1e9    # seconds per Gyr
t_Hubble = 14.4 * Gyr_s # Hubble time

### Problem 3(a) - Plotting the integrand (1 pt)

Before evaluating an integral, it is a good idea to evalulate the integrand.

Plot the integrand
$$ \frac{e^{29/19}\left(1 + \frac{121}{304}e^2\right)^{1181/2299}}{(1-e^2)^{3/2}} $$
as a function of eccentricity $e$, from $e = 0.01$ to $e = 0.99$. Plot the x-axis as linear, and the y-axis as logarithmic. 

In the markdown cell after the code cell, explain: What do we see? Where might we expect problems in evaluating this integral?

In [ ]:
# Code for 3a here


**Explanation for 3a here**


### Problem 3(b) - Writing a sympson integrator for the Peter's integral (1 pt)

Write a function that calculates Peter's gravitational wave merger time,
$$ T_{\rm merge} = \frac{12}{19}\frac{c_0^4}{\beta} \int_0^{e_0} \frac{e^{29/19}\left(1 + \frac{121}{304}e^2\right)^{1181/2299}}{(1-e^2)^{3/2}}\,de$$
with the Simpson method, assuming our initial binary parameters. Have as input values the number of subintervals $N$ you break your integrand up in your numerical approximation, and the initial eccentricity, $e_0$. Have the subinterval sizes on the x-axis be $\Delta e = e_0/N$. Have your output be the output merger time given in years.

In [2]:
# Code for 3b here


### Problem 3(c) - Determine step-size for accurate integral (1 pt)

Evaluate the merger time, for 30 values of $N$ spaced logarithmically from $10^2$ to $10^5$, for values of $e_0 = 0.01$, $e_0=0.5$, and $e_0 = 0.99$. Display each integral in a seperate subplot, labeling the $e_0$ as an f-string either in the subplot title, using `ax.text`, or in a legend caption within the plot. Plot the x-axis as logarithmic, but the y-axis is linear. Divide the merger time you evaulated at each $N$, by the merger time you evaluated at $N=10^5$, so we can clearly see how the merger time asymptotes at large $N$.

How does the error in the numerical evaluation of $T_{\rm merge}$ depend on $e_0$?

In [3]:
# Code for 3c here


**Explanation for 3c here**


### Problem 3(d) - Calculate how gravitational wave merger time depends on initial eccentricity (1 pt)

Evaluate the Peter's merger time integral using the $N$ you determined in the last section to return an accurate integral. Evaluate the initial eccentricity $e_0$ between $0.01$ and $0.99$, with $10^3$ steps spaced linearly between these two values. 

Create a plot that calculates how the gravitational wave merger time depends on the binary initial eccentricity, plotting $T_{\rm merge}$ versus $e_0$. Make sure the y-axis is logarithmic, and the merger time is plotted in years. Compare your result to the Hubble time, $14.4$ Gyr, or the rough age of the universe, using `ax.hlines`. Label both the merger time and hubble time, using `ax.text` or with a legend. In the markdown cell below, discuss the initial eccentricities required to have a neutron star binary merge over the age of the observable universe.

In [ ]:
# Code for 3d here


**Explanation for 3d here**
